# Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from data import MET_Data
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_score

In [45]:
def get_from_csv(csv_dir, patch_data):
    arbor_types = {"exc": (("apical", 2), ("basal", 3)), "inh": (("axon", 0), ("basal", 1))}
    morphs = []
    for (sp_id, neuron_class) in zip(patch_data["specimen_id"], patch_data["class"]):
        blank = None
        for (arbor, index) in arbor_types[neuron_class]:
            csv_path = f"../data/morph/{csv_dir}/{sp_id}_{arbor}_histogram.csv"
            loaded = np.loadtxt(csv_path, dtype = "float32", delimiter = " ")
            if blank is None:
                blank = np.zeros(loaded.shape + (4,))
            blank[..., index] = loaded
        morphs.append(blank)
    morphs = np.stack(morphs, 0)
    return morphs

def get_from_data(key, patch_data):
    return patch_data[key]

In [3]:
data_keys = {"logcpm": "logcpm", "pca-ipfx": "pca-ipfx", "arbors": "arbors", "morph": "morphometric"}
met_data = MET_Data("../data/neurons.hdf5", **data_keys)
for form in ["logcpm", "pca-ipfx", "arbors", "morph"]:
    met_data.cache_data(form)
patch_data = met_data.query(formats = [("logcpm", "arbors", "morph")])
num_samples = len(patch_data["logcpm"])

Caching logcpm...
Caching pca-ipfx...
Caching arbors...
Caching morph...


# Results

In [46]:
featurizations = {
    "120_1_4": [(get_from_csv, "120_1")],
    "120_4_4": [(get_from_data, "arbors")],
    "120_8_4": [(get_from_csv, "120_8")],
    "120_16_4": [(get_from_csv, "120_16")],
    "120_32_4": [(get_from_csv, "120_32")],
    "120_60_4": [(get_from_csv, "120_60")],
    "120_120_4": [(get_from_csv, "120_120")],
    "irred_120_1": [(get_from_csv, "irred_120_4")],
    "irred_120_4": [(get_from_csv, "irred_120_4")],
    "irred_120_8": [(get_from_csv, "irred_120_8")],
    "irred_120_16": [(get_from_csv, "irred_120_16")],
    "irred_120_32": [(get_from_csv, "irred_120_32")],
    "irred_120_60": [(get_from_csv, "irred_120_60")],
    "irred_120_120": [(get_from_csv, "irred_120_120")],
    "120_1_concat": [(get_from_csv, "120_1"), (get_from_csv, "irred_120_1")],
    "120_4_concat": [(get_from_data, "arbors"), (get_from_csv, "irred_120_4")],
    "120_8_concat": [(get_from_csv, "120_8"), (get_from_csv, "irred_120_8")],
    "120_16_concat": [(get_from_csv, "120_16"), (get_from_csv, "irred_120_16")],
    "120_32_concat": [(get_from_csv, "120_32"), (get_from_csv, "irred_120_32")],
    "120_60_concat": [(get_from_csv, "120_60"), (get_from_csv, "irred_120_60")],
    "120_120_concat": [(get_from_csv, "120_120"), (get_from_csv, "irred_120_120")]
    # "morphometric": [(get_from_data, "morph")],
    # "120_4_4+morph": [(get_from_data, "arbors"), (get_from_data, "morph")],
    # "120_4_4+morph+irred": [(get_from_data, "arbors"), (get_from_data, "morph"), (get_from_csv, "irred_120_4")]
}

In [47]:
scores = {}

In [48]:
for (feature, params) in featurizations.items():
    if feature in scores: 
        continue
    print(f"Running {feature}                  ", end = "\r")
    outputs = [func(arg, patch_data) for (func, arg) in params]
    X = np.concatenate([arr.reshape(arr.shape[0], -1) for arr in outputs], -1)
    X = np.nan_to_num(X)
    classifier = RandomForestClassifier()
    scores[feature] = cross_val_score(classifier, X, patch_data["cluster_label"], cv = 10).mean()


/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


...

/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


/Users/ian.convy/miniconda3/envs/cpl/lib/python3.8/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


In [34]:
for (feature, score) in scores.items():
    print(f"{feature}: {score:.4f}")

120_4_4: 0.3913
irred_120_4: 0.3565
120_4_concat: 0.3891
morphometric: 0.4522
120_4_4+morph: 0.4293
120_4_4+morph+irred: 0.4326
120_8_4: 0.3913
irred_120_8: 0.3652
120_8_concat: 0.4033
irred_120_60: 0.3109
irred_120_120: 0.2848


In [49]:
for (feature, score) in scores.items():
    print(f"{feature}: {score:.4f}")

120_1_4: 0.4130
120_4_4: 0.3935
120_8_4: 0.4000
120_16_4: 0.3891
120_32_4: 0.3891
120_60_4: 0.3609
120_120_4: 0.3533
irred_120_1: 0.3554
irred_120_4: 0.3576
irred_120_8: 0.3435
irred_120_16: 0.3630
irred_120_32: 0.3543
irred_120_60: 0.3185
irred_120_120: 0.2848
120_1_concat: 0.3989
120_4_concat: 0.3935
120_8_concat: 0.4065
120_16_concat: 0.4087
120_32_concat: 0.4098
120_60_concat: 0.3576
120_120_concat: 0.3424
